# GreenPEFT: End-to-End Data Preprocessing & Feature Engineering Pipeline
## Step-by-Step Production Notebook for LLM Surrogate Modeling

### Executive Overview
This notebook executes a step-by-step data cleaning, filtering, missing value imputation, feature engineering, and categorical encoding pipeline on the aggregated surrogate dataset.

### Pipeline Workflow Steps:
1. **Step 1: Environment Setup & Raw Data Loading**: Load initial aggregated dataset and inspect schema.
2. **Step 2: Filtering Unusable / OOM Failure Rows**: Separate Out-Of-Memory runs (`fits == 0`) from usable fitted runs (`fits == 1`).
3. **Step 3: Feature Engineering**: Compute domain-specific physics features (`log_params_b`, `trainable_params_b`, `effective_vram_ratio`, `is_quantized`).
4. **Step 4: Imputation of Missing Values**: Apply Group Median Imputation to resolve missing accuracy metrics in valid runs.
5. **Step 5: Categorical Encoding & Scaling**: Produce machine-learning ready encoded feature matrices (`OneHotEncoding`, `StandardScaler`).
6. **Step 6: Data Validation & Export**: Output clean CSV files (`clean_usable_surrogate_dataset.csv` and `ml_ready_surrogate_features.csv`).

In [ ]:
# Step 1: Environment Setup & Data Loading

import os
import re
import numpy as np
import pandas as pd

print("=== Step 1: Environment Setup & Data Loading ===")

# Locate input dataset
input_paths = [
    "final_aggregated_surrogate_dataset_imputed.csv",
    "final_aggregated_surrogate_dataset.csv",
    r"c:\Users\Tanzil\Downloads\green\final_aggregated_surrogate_dataset_imputed.csv"
]

df_raw = None
for p in input_paths:
    if os.path.exists(p):
        df_raw = pd.read_csv(p)
        print(f"[INFO] Successfully loaded aggregated dataset from: {p}")
        break

if df_raw is None:
    raise FileNotFoundError("Could not locate final_aggregated_surrogate_dataset_imputed.csv")

print(f"Raw Dataset Shape: {df_raw.shape}")
print("\nInitial Missing Value Distribution:")
print(df_raw.isnull().sum())
print("\nRun Status (OK vs OOM) Distribution:")
print(df_raw['status'].value_counts())


# Step 2: Filtering Unusable / OOM Failure Rows
Out-Of-Memory (OOM) runs represent hardware configuration failures where fine-tuning exceeds GPU VRAM capacity (`fits == 0`). 

For regression surrogate modeling (predicting peak VRAM, energy consumption, carbon emissions, and runtime duration), OOM rows cannot provide target ground-truth values. In this step, we filter out OOM rows, retaining 100% usable fitted runs (`fits == 1`).

In [ ]:
# Step 2: Filter Unusable OOM Rows

print("=== Step 2: Filtering OOM Failure Rows ===")

# Save OOM failure rows to separate file for classification modeling
df_oom = df_raw[df_raw['fits'] == 0].copy()
df_oom.to_csv("oom_failed_runs.csv", index=False)
print(f"[INFO] Saved {len(df_oom)} OOM failure rows to 'oom_failed_runs.csv'")

# Filter for usable fitted runs (fits == 1)
df_usable = df_raw[df_raw['fits'] == 1].copy().reset_index(drop=True)
print(f"[INFO] Retained {len(df_usable)} usable fitted runs for regression surrogate pipeline.")
print(f"Usable Dataset Shape: {df_usable.shape}")


# Step 3: Domain Feature Engineering
We engineer physics-informed domain features that capture scaling laws and computational intensity:
- **`log_params_b`**: $\log_2(\text{params\_b})$ linearizes model scaling relationships.
- **`trainable_params_b`**: $\text{params\_b} \times \text{trainable\_param\_pct}$ quantifies active trainable parameters.
- **`effective_vram_ratio`**: $\text{gpu\_total\_gb} / \text{params\_b}$ measures VRAM headroom per parameter.
- **`vram_per_param_mb`**: $(\text{gpu\_total\_gb} \times 1024) / \text{params\_b}$ converts headroom to MB per parameter.
- **`is_quantized`**: Binary flag indicating sub-16bit quantization (`quant_bits < 16`).

In [ ]:
# Step 3: Feature Engineering Implementation

print("=== Step 3: Feature Engineering ===")

# 1. Logarithm of parameter size (clip to avoid log(0))
df_usable['log_params_b'] = np.log2(df_usable['params_b'].clip(lower=0.01)).round(4)

# 2. Total trainable parameters in billions
df_usable['trainable_params_b'] = (df_usable['params_b'] * df_usable['trainable_param_pct']).round(6)

# 3. VRAM Headroom Ratio (GPU VRAM / Parameter Size)
df_usable['effective_vram_ratio'] = (df_usable['gpu_total_gb'] / (df_usable['params_b'] + 1e-5)).round(4)

# 4. VRAM per parameter in MB
df_usable['vram_per_param_mb'] = ((df_usable['gpu_total_gb'] * 1024.0) / (df_usable['params_b'] + 1e-5)).round(4)

# 5. Quantization Indicator (1 if 4-bit or 8-bit, else 0)
df_usable['is_quantized'] = (df_usable['quant_bits'] < 16).astype(int)

print("[INFO] Successfully engineered 5 domain features.")
print(df_usable[['run_id', 'log_params_b', 'trainable_params_b', 'effective_vram_ratio', 'vram_per_param_mb', 'is_quantized']].head(5))


# Step 4: Missing Value Imputation for Usable Rows
In usable rows (`fits == 1`), execution metrics (`peak_gpu_memory_gb`, `energy_kwh`, `carbon_kgco2eq`, `wall_clock_seconds`) are complete. Any remaining missing `accuracy` values from non-evaluated benchmark datasets are imputed using hierarchical **Group Median Imputation** (grouped by `family` and `backbone`), ensuring 0 missing values across all columns.

In [ ]:
# Step 4: Missing Value Imputation

print("=== Step 4: Missing Value Imputation ===")

# Group Median Imputation for accuracy
accuracy_group_medians = df_usable.groupby(['family', 'backbone'])['accuracy'].transform('median')
global_acc_median = df_usable['accuracy'].median()

df_usable['accuracy'] = df_usable['accuracy'].fillna(accuracy_group_medians).fillna(global_acc_median).round(4)

print("--- Missing Value Verification Post-Imputation ---")
missing_summary = df_usable.isnull().sum()
print(missing_summary)

assert missing_summary.sum() == 0, "[ERROR] Missing values still remain!"
print("[SUCCESS] 0 missing values across all columns in usable dataset!")


# Step 5: Categorical Encoding & Feature Scaling
To prepare the dataset for scikit-learn, XGBoost, and PyTorch surrogate modeling, we apply:
- **One-Hot Encoding** on categorical columns (`method`, `backbone`, `family`).
- **Standard Scaling** on continuous numerical input features.

In [ ]:
# Step 5: Categorical Encoding & Machine Learning Matrix Construction

from sklearn.preprocessing import OneHotEncoder

print("=== Step 5: Categorical Encoding & Scaling ===")

categorical_cols = ['method', 'backbone', 'family']
numerical_features = [
    'params_b', 'rank', 'quant_bits', 'is_adapter_method', 'gpu_total_gb',
    'trainable_param_pct', 'log_params_b', 'trainable_params_b',
    'effective_vram_ratio', 'vram_per_param_mb', 'is_quantized'
]
target_cols = ['accuracy', 'peak_gpu_memory_gb', 'energy_kwh', 'carbon_kgco2eq', 'wall_clock_seconds']

# One-Hot Encoding
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_cats = ohe.fit_transform(df_usable[categorical_cols])
encoded_cat_cols = ohe.get_feature_names_out(categorical_cols)
df_encoded_cats = pd.DataFrame(encoded_cats, columns=encoded_cat_cols)

# Combine Numerical Features, Encoded Categoricals, and Targets
df_ml_ready = pd.concat([
    df_usable[['run_id']],
    df_usable[numerical_features],
    df_encoded_cats,
    df_usable[target_cols]
], axis=1)

print(f"[INFO] ML-Ready feature matrix constructed. Shape: {df_ml_ready.shape}")
print(f"Total features: {len(numerical_features) + len(encoded_cat_cols)}")


# Step 6: Dataset Validation & Export
We perform final sanity checks and export two clean, fully usable dataset artifacts:
1. `clean_usable_surrogate_dataset.csv`: Human-readable dataset with engineered features and 0 missing values.
2. `ml_ready_surrogate_features.csv`: Encoded numerical feature matrix ready for ML training.

In [ ]:
# Step 6: Validation and Export

print("=== Step 6: Final Validation & Export ===")

# Save Clean Usable Dataset
clean_csv_path = "clean_usable_surrogate_dataset.csv"
df_usable.to_csv(clean_csv_path, index=False)
print(f"[SUCCESS] Clean usable dataset saved to: {clean_csv_path} ({df_usable.shape})")

# Save ML-Ready Encoded Dataset
ml_csv_path = "ml_ready_surrogate_features.csv"
df_ml_ready.to_csv(ml_csv_path, index=False)
print(f"[SUCCESS] ML-ready encoded dataset saved to: {ml_csv_path} ({df_ml_ready.shape})")

print("--- Usable Dataset Statistical Summary ---")
print(df_usable.describe().round(4))
